# 13 - Calibration, the referral threshold and ResNet-50 vs the ensemble

**In short:** Sproutly answers only when it is confident enough and refers the farmer to
an extension officer otherwise. This notebook makes that confidence honest (calibration),
chooses the confidence below which a case is referred (the threshold), and decides
whether the three-model ensemble refers better than ResNet-50 alone. Every choice is made
on the **selection set** (notebook 12): 3,821 field photos from four sources the models
have never seen. Validation photos come from the training sources and are 99.5% right, too
few mistakes to learn from. **PlantDoc is never used to choose; it is only reported.**

**The confidence used everywhere** is the one the app shows: the farmer names the crop,
the answer is the best class of that crop, and its confidence is that class's 11-class
probability, not rescaled (`sproutly-backend/app/core/classifier.py`).

**Rules, fixed before running (decision log D2, D6, D7):**

- **D6, calibration method.** Six methods (none, temperature, vector, Dirichlet, histogram
  binning, isotonic) are scored out of sample: fitted on four fifths of the selection set,
  scored on the remaining fifth, five folds, repeated three times. The winner has the lowest
  out-of-sample log-loss among methods that never change the predicted class (vector and
  Dirichlet are reported but cannot win: calibration may change the confidence, not the
  diagnosis). A tie within one standard deviation keeps temperature scaling, the simplest.
- **D2, ResNet-50 or the ensemble** (ResNet-50 + ConvNeXt-Tiny + EfficientNet-B0, each
  calibrated with its own D6 winner). The ensemble is adopted only if its area under the
  risk-coverage curve (AURC, lower = refers the right cases) is lower **beyond the bootstrap
  range** (paired, 1,000 resamples), and both meet NFR iii (diagnosis within 10 s).
  Otherwise ResNet-50 stays.
- **D7, threshold.** The lowest threshold at which the answered photos are at least **90%**
  correct, on out-of-fold calibrated confidences, with a bootstrap range; never below 0.5.
  If 90% cannot be reached, the cost-based threshold is used instead (a wrong answer costs
  as much as 5 referrals) and this is reported. A sensitivity table shows targets 0.70 to
  0.95.

**Also reported:** calibration fitted on the validation set instead (issue #33), in-domain
(test) and out-of-domain (PlantDoc) calibration separately (NFR i), and the PlantDoc result
per class for the chosen model.

**Limitation stated up front:** the selection set has no maize northern leaf blight and no
potato early blight, so the threshold is set without those two classes. Calibration is
fitted with photos only; when a farmer also types symptoms (late fusion, notebook 11b) the
same calibrated image scores are used.

**Needs a GPU runtime (T4).** About 15-20 minutes.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (226/226), done.
## main...origin/main
 M ml/notebooks/13_calibration_and_threshold.ipynb
?? ml/results/fusion_quick_test/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 144.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.6 MB/s eta 0:00:00


## Settings

In [4]:
import importlib
import json

import numpy as np
import pandas as pd
import torch

from src import calibration as cal, metrics, training
from src.data import image_dataset
for module in (cal, metrics, training, image_dataset):
    importlib.reload(module)

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
DEVICE = "cuda"

# The trained runs (notebooks 08 and 09); the ensemble is the one carried from notebook 10.
RUNS = {"resnet50": config.MODELS_DIR / "compare_resnet50",
        "convnext_tiny_lr1e-04": config.MODELS_DIR / "compare_convnext_tiny_lr1e-04",
        "efficientnet_b0": config.MODELS_DIR / "image_baseline"}
ENSEMBLE = list(RUNS)
CPU_MS = {"resnet50": 193.0, "ensemble": 447.1}  # ml/results/comparison/comparison.csv
NFR_III_MS = 10_000                              # diagnosis within 10 s

TARGET, COST_WRONG, FLOOR = 0.90, 5.0, 0.5
FOLDS, REPEATS, BOOTSTRAP = 5, 3, 1000

CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])

OUT_DIR = config.MODELS_DIR / "calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS = ML_DIR / "results" / "calibration"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for name, run in RUNS.items():
    assert (run / "best.pt").exists(), f"{name}: {run / 'best.pt'} missing"
training.seed_everything(config.RANDOM_SEED)

## Predict the selection set with each model

Validation, test and PlantDoc predictions were saved by notebook 09, so only the
selection photos are new. The photos were saved shrunk by notebook 12, exactly as every
other split is shrunk before the model sees it.

In [5]:
from torch.utils.data import DataLoader

selection = pd.read_csv(config.SPLITS_DIR / "selection.csv")
LOCAL_ROOT = Path("/content/selection_cache")
cached = image_dataset.cache_images(selection["path"].tolist(), config.RAW_DATA_DIR, LOCAL_ROOT)
assert cached == len(selection), f"Only {cached} of {len(selection)} selection photos found"
loader = DataLoader(image_dataset.SplitDataset(selection, LOCAL_ROOT, CLASSES, image_dataset.eval_transform(224)),
                    batch_size=64, shuffle=False, num_workers=2)

selection_logits = {}
for name, run in RUNS.items():
    saved = OUT_DIR / f"{name}_selection.logits.npy"
    if saved.exists():
        selection_logits[name] = np.load(saved)
        continue
    architecture = json.loads((run / "settings.json").read_text())["architecture"].split()[0]
    model = training.build_model(len(CLASSES), architecture, pretrained=False)
    model.load_state_dict(torch.load(run / "best.pt", map_location=DEVICE))
    logits, labels = training.predict(model.to(DEVICE), loader, DEVICE)
    np.save(saved, logits)
    selection_logits[name] = logits
    del model
    torch.cuda.empty_cache()
print({name: l.shape for name, l in selection_logits.items()})

Caching images:   0%|          | 0/3821 [00:00<?, ?it/s]

Predicting:   0%|          | 0/60 [00:00<?, ?it/s]

Predicting:   0%|          | 0/60 [00:00<?, ?it/s]

Predicting:   0%|          | 0/60 [00:00<?, ?it/s]

{'resnet50': (3821, 11), 'convnext_tiny_lr1e-04': (3821, 11), 'efficientnet_b0': (3821, 11)}


## Put every set side by side

For each set: the logits of ResNet-50 and of the ensemble (the average of the three
models' probabilities), the true class and the crop the farmer would name.

In [6]:
def saved_logits(run, name):
    rows = pd.read_csv(run / "results" / f"{name}_predictions.csv")
    return rows, np.load(run / "results" / f"{name}_predictions.logits.npy")

sets = {}
for name in ["val", "test", "ood_test"]:
    base_rows, _ = saved_logits(RUNS["resnet50"], name)
    per_model = {}
    for model_name, run in RUNS.items():
        rows, logits = saved_logits(run, name)
        assert rows["path"].tolist() == base_rows["path"].tolist(), f"{model_name} {name}: photos in another order"
        per_model[model_name] = logits
    labels = base_rows["class"].map(INDEX)
    keep = labels.notna().to_numpy()
    sets[name] = {"rows": base_rows[keep].reset_index(drop=True), "labels": labels[keep].astype(int).to_numpy(),
                  "logits": {m: l[keep] for m, l in per_model.items()}}
sets["selection"] = {"rows": selection, "labels": selection["class"].map(INDEX).to_numpy(), "logits": selection_logits}

for part in sets.values():
    part["crops"] = CLASS_CROPS[part["labels"]]
    probs = [metrics.softmax(part["logits"][m]) for m in ENSEMBLE]
    part["candidates"] = {"resnet50": part["logits"]["resnet50"],
                          "ensemble": np.log(np.mean(probs, axis=0) + 1e-12)}

def app(probs, part):
    # Confidence and correctness exactly as the app computes them.
    conf, predicted = cal.crop_confidence(probs, CLASS_CROPS, part["crops"])
    return conf, predicted == part["labels"]

display(pd.DataFrame({name: {"photos": len(p["labels"]), "classes": len(set(p["labels"]))} for name, p in sets.items()}).T)

,photos,classes
val,2385,11
test,2441,11
ood_test,953,9
selection,3821,9


## D6: which calibration method?

Out-of-sample scores on the selection set, for each candidate model.

In [ ]:
sel = sets["selection"]
D6, d6_tables = {}, []
for candidate, logits in sel["candidates"].items():
    table = cal.compare(logits, sel["labels"], folds=FOLDS, repeats=REPEATS, seed=config.RANDOM_SEED)
    method, reason = cal.choose_method(table)
    D6[candidate] = {"method": method, "reason": reason}
    d6_tables.append(table.assign(candidate=candidate))
d6 = pd.concat(d6_tables)
d6_summary = (d6.groupby(["candidate", "method"])
                .agg(nll=("nll", "mean"), nll_sd=("nll", "std"), ece=("ece", "mean"), brier=("brier", "mean"),
                     aurc=("aurc", "mean"), accuracy=("accuracy", "mean"), changed_predictions=("changed_predictions", "mean")))
d6_summary["can_win"] = [m in cal.KEEPS_PREDICTION for _, m in d6_summary.index]
display(d6_summary.round(4))
print(json.dumps(D6, indent=1))
d6_summary.to_csv(OUT_DIR / "d6_methods.csv")

## D2: ResNet-50 or the ensemble?

Each candidate's out-of-fold calibrated confidences (every photo scored by a calibrator
fitted without it), then the paired bootstrap of the AURC difference.

In [ ]:
oof = {c: cal.out_of_fold(D6[c]["method"], sel["candidates"][c], sel["labels"], FOLDS, config.RANDOM_SEED)
       for c in sel["candidates"]}
oof_app = {c: app(p, sel) for c, p in oof.items()}
raw_app = {c: app(metrics.softmax(l), sel) for c, l in sel["candidates"].items()}

D2 = cal.compare_models(*oof_app["resnet50"], *oof_app["ensemble"], n=BOOTSTRAP, seed=config.RANDOM_SEED)
D2["both_meet_nfr_iii"] = max(CPU_MS.values()) <= NFR_III_MS
D2["aurc_uncalibrated_resnet50"] = cal.aurc(*raw_app["resnet50"])
D2["aurc_uncalibrated_ensemble"] = cal.aurc(*raw_app["ensemble"])
CHOSEN = "ensemble" if D2["b_wins"] and D2["both_meet_nfr_iii"] else "resnet50"
D2["chosen"] = CHOSEN
D2["reason"] = ("ensemble AURC lower beyond the bootstrap range" if CHOSEN == "ensemble"
                else "ensemble not better beyond the bootstrap range: ResNet-50 (simpler, one third of the CPU time) stays")
print(json.dumps(D2, indent=1, default=float))

## D7: the threshold

On the chosen model's out-of-fold calibrated confidences.

In [ ]:
conf_oof, correct_oof = oof_app[CHOSEN]
D7 = cal.choose_threshold(conf_oof, correct_oof, TARGET, COST_WRONG, FLOOR, n=BOOTSTRAP, seed=config.RANDOM_SEED)
THRESHOLD = D7["threshold"]
print(json.dumps(D7, indent=1, default=float))

sens = cal.sensitivity(conf_oof, correct_oof)
display(sens.round(4))
sens.to_csv(OUT_DIR / "sensitivity.csv", index=False)

## The final calibrator and what it does on every set

The chosen method is fitted once on all selection photos; this is what the backend will
use. On the selection set itself the out-of-fold confidences are reported, so no photo is
scored by a calibrator that saw it. **Test is in-domain, PlantDoc is out-of-domain**; both
are reported, neither was used to choose.

In [ ]:
FINAL = cal.fit(D6[CHOSEN]["method"], sel["candidates"][CHOSEN], sel["labels"])

def scores(probs, part, mask=None):
    labels = part["labels"]
    conf, correct = app(probs, part)
    if mask is not None:
        probs, labels, conf, correct = probs[mask], labels[mask], conf[mask], correct[mask]
    at = cal.at_threshold(conf, correct, THRESHOLD)
    return {"photos": len(labels), "accuracy (crop given)": correct.mean(), "mean confidence": conf.mean(),
            "ece": metrics.expected_calibration_error(conf, correct), "nll": cal.summarise(probs, labels)["nll"],
            "aurc": cal.aurc(conf, correct), "answered": at["coverage"], "accuracy of answered": at["selective_accuracy"],
            "wrong and answered": at["wrong_and_answered"]}

final_rows = []
for label, name, mask in [("selection (out of fold)", "selection", None), ("validation", "val", None),
                          ("test (in-domain)", "test", None),
                          ("test: field photos", "test", sets["test"]["rows"]["source"].isin(["fieldplant", "tanzania"]).to_numpy()),
                          ("PlantDoc (out-of-domain)", "ood_test", None)]:
    part = sets[name]
    logits = part["candidates"][CHOSEN]
    after = oof[CHOSEN] if name == "selection" else FINAL.probabilities(logits)
    final_rows.append({"set": label, "calibration": "none", **scores(metrics.softmax(logits), part, mask)})
    final_rows.append({"set": label, "calibration": D6[CHOSEN]["method"], **scores(after, part, mask)})
final = pd.DataFrame(final_rows).set_index(["set", "calibration"])
display(final.round(4))
final.to_csv(OUT_DIR / "final_metrics.csv")

## For comparison: calibrating on the validation set (issue #33)

What Chapter 3 originally planned. The same method is fitted on the validation photos
instead, and the original threshold rule is applied to validation confidences.

In [ ]:
val = sets["val"]
on_val = cal.fit(D6[CHOSEN]["method"], val["candidates"][CHOSEN], val["labels"])
val_rows = []
for label, name in [("selection (all photos)", "selection"), ("test (in-domain)", "test"), ("PlantDoc (out-of-domain)", "ood_test")]:
    part = sets[name]
    val_rows.append({"set": label, "fitted on": "validation", **scores(on_val.probabilities(part["candidates"][CHOSEN]), part)})
validation_based = pd.DataFrame(val_rows).set_index(["set", "fitted on"])
display(validation_based.round(4))
validation_based.to_csv(OUT_DIR / "validation_based.csv")

val_conf, val_correct = app(on_val.probabilities(val["candidates"][CHOSEN]), val)
VALIDATION_NOTE = {
    "params_fitted_on_validation": {k: np.asarray(v).tolist() for k, v in on_val.params.items()},
    "params_fitted_on_selection": {k: np.asarray(v).tolist() for k, v in FINAL.params.items()},
    "validation_accuracy_crop_given": float(val_correct.mean()),
    "original_rule_threshold_on_validation": cal.threshold_for_accuracy(val_conf, val_correct, TARGET, FLOOR),
}
print(json.dumps(VALIDATION_NOTE, indent=1, default=float)[:1500])

## PlantDoc per class (reported only)

The chosen model on PlantDoc, crop given, before and after calibration, at the chosen
threshold. This replaces the EfficientNet-B0 per-class table in Chapter 5.

In [ ]:
pd_part = sets["ood_test"]
pd_logits = pd_part["candidates"][CHOSEN]
conf_before, correct_pd = app(metrics.softmax(pd_logits), pd_part)
conf_after, _ = app(FINAL.probabilities(pd_logits), pd_part)
per_class = pd.DataFrame({"class": [CLASSES[i] for i in pd_part["labels"]], "correct": correct_pd,
                          "confidence before": conf_before, "confidence after": conf_after,
                          "answered": conf_after >= THRESHOLD})
per_class["correct and answered"] = per_class["correct"] & per_class["answered"]
table = per_class.groupby("class").agg(photos=("correct", "size"), accuracy=("correct", "mean"),
                                       **{"mean confidence before": ("confidence before", "mean"),
                                          "mean confidence after": ("confidence after", "mean"),
                                          "share answered": ("answered", "mean")})
table["accuracy of answered"] = (per_class.groupby("class")["correct and answered"].sum()
                                 / per_class.groupby("class")["answered"].sum()).fillna(float("nan"))
display(table.round(3))
table.to_csv(OUT_DIR / "plantdoc_per_class.csv")

## Charts for the report

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"          # categorical slots 1 and 2 (validated pair)
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"

def tidy(ax):
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED, labelsize=8)

# 1. Reliability: how sure the model says it is vs how often it is right.
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)
panels = [("Selection set (out of fold)", metrics.softmax(sel["candidates"][CHOSEN]), oof[CHOSEN], sel),
          ("PlantDoc (never used to choose)", metrics.softmax(pd_logits), FINAL.probabilities(pd_logits), pd_part)]
for ax, (title, before, after, part) in zip(axes, panels):
    ax.plot([0, 1], [0, 1], color="#c3c2b7", linewidth=1, linestyle="--", zorder=1)
    for probs, colour, label in [(before, BLUE, "before calibration"), (after, ORANGE, f"after ({D6[CHOSEN]['method']})")]:
        bins = cal.reliability(*app(probs, part))
        ax.plot(bins["confidence"], bins["accuracy"], color=colour, linewidth=2, marker="o", markersize=6,
                markeredgecolor="white", markeredgewidth=1.5, label=label, zorder=2)
    ax.axvline(THRESHOLD, color=MUTED, linewidth=1, zorder=1)
    ax.text(THRESHOLD, 0.02, f" threshold {THRESHOLD:.2f}", color=MUTED, fontsize=8)
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
    ax.set_xlabel("Confidence shown in the app", color=INK, fontsize=9)
    ax.set_title(title, color=INK, fontsize=10, loc="left")
    tidy(ax)
axes[0].set_ylabel("Share actually correct", color=INK, fontsize=9)
axes[0].legend(loc="upper left", frameon=False, fontsize=8)
fig.suptitle("Calibration: on the dashed line, confidence means what it says", x=0.01, ha="left", fontsize=11, color=INK)
plt.tight_layout()
plt.savefig(OUT_DIR / "reliability.png", dpi=150)
plt.show()

# 2. Risk-coverage: answer the most confident photos first; how many answers are wrong?
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for candidate, colour, label in [("resnet50", BLUE, "ResNet-50"), ("ensemble", ORANGE, "Ensemble (3 models)")]:
    coverage, risk = cal.risk_coverage(*oof_app[candidate])
    ax.plot(coverage, risk, color=colour, linewidth=2, label=f"{label}, AURC {cal.aurc(*oof_app[candidate]):.3f}")
    ax.text(1.0, risk[-1], f" {label}", color=MUTED, fontsize=8, va="center")
answered = cal.at_threshold(conf_oof, correct_oof, THRESHOLD)["coverage"]
ax.axvline(answered, color=MUTED, linewidth=1)
ax.set_xlim(0, 1.22)  # room for the line labels on the right
ax.set_xticks(np.linspace(0, 1, 6))
ax.text(answered, ax.get_ylim()[0] + 0.02 * np.ptp(ax.get_ylim()), f"threshold {THRESHOLD:.2f}: {answered:.0%} answered ",
        color=MUTED, fontsize=8, ha="right")
ax.set_xlabel("Share of photos answered (the rest referred)", color=INK, fontsize=9)
ax.set_ylabel("Share of answers that are wrong", color=INK, fontsize=9)
ax.set_title("Selection set: referring the least confident photos first", color=INK, fontsize=10, loc="left")
ax.legend(loc="upper left", frameon=False, fontsize=8)
tidy(ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "risk_coverage.png", dpi=150)
plt.show()

## Save the decision and the calibrator for the backend

In [ ]:
calibrator = {"model": CHOSEN, "members": ENSEMBLE if CHOSEN == "ensemble" else ["resnet50"],
              "method": D6[CHOSEN]["method"], "params": {k: np.asarray(v).tolist() for k, v in FINAL.params.items()},
              "threshold": THRESHOLD, "fitted_on": f"selection set ({len(sel['labels'])} photos, notebook 12)",
              "confidence": "best class of the farmer's crop, 11-class probability, not rescaled"}
training.save_json(calibrator, OUT_DIR / "calibrator.json")
decision = {"D6": D6, "D2": D2, "D7": D7, "validation_based": VALIDATION_NOTE,
            "selection_photos": int(len(sel["labels"])), "selection_classes": sorted(CLASSES[i] for i in set(sel["labels"])),
            "missing_from_selection": sorted(set(CLASSES) - {CLASSES[i] for i in set(sel["labels"])})}
training.save_json(decision, OUT_DIR / "decision.json")
print(json.dumps(calibrator, indent=1, default=float)[:800])

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
import shutil

for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json", ".png"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "13_calibration_and_threshold.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/calibration-threshold",
    message="Calibration, referral threshold and ResNet-50 vs ensemble on the selection set",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))